# Dataset Inventory & Data Quality

Notebook này quét toàn bộ dataset hiện có trong `data/`, đối chiếu practice và redacted, rồi chỉ ra các điểm đáng chú ý cho team:

- số trip, số frame, độ dài, FPS
- mức độ đầy đủ của từng modality
- trip nào có/không có driver label hoặc risk ground truth
- coverage của image/depth/label/calibration

In [2]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "AGENTS.md").exists():
    if (PROJECT_ROOT.parent / "AGENTS.md").exists():
        PROJECT_ROOT = PROJECT_ROOT.parent
    else:
        raise FileNotFoundError("Cannot locate project root from the current notebook working directory.")

NOTEBOOK_HELPERS = PROJECT_ROOT / "notebooks"
if str(NOTEBOOK_HELPERS) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_HELPERS))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import fleetiq_notebook_utils as nb

plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 160)

In [3]:
inventory = nb.build_dataset_inventory()
inventory

,trip_id,dataset_name,frame_count,duration_sec,fps_estimate,driver_labels_available,risk_ground_truth_available,trip_aggregate_available,driver_summary_available,event_log_count,target_count_mean,target_count_max,events_active_max,left_images,right_images,driver_images,depth_files,calib_files,label_files,left_image_coverage,right_image_coverage,driver_image_coverage,depth_coverage,label_coverage
0,T01-Sample,practice,600,29.95,20.0,True,True,True,True,1,11.881667,17,1,600,600,600,120,600,600,1.000000,1.0,1.0,0.2,1.0
1,T02-Sample,practice,600,29.95,20.0,True,True,True,True,1,7.555000,14,1,600,600,600,120,600,600,1.000000,1.0,1.0,0.2,1.0
2,T03-Sample,practice,600,29.95,20.0,True,True,True,True,1,0.958333,2,1,600,600,600,120,600,600,1.000000,1.0,1.0,0.2,1.0
3,T04-Sample,practice,600,29.95,20.0,True,True,True,True,1,2.996667,9,1,600,600,600,120,600,600,1.000000,1.0,1.0,0.2,1.0
4,T05-Sample,practice,600,29.95,20.0,True,True,True,True,2,1.608333,3,2,600,600,600,120,600,600,1.000000,1.0,1.0,0.2,1.0
5,T06-Sample,practice,600,29.95,20.0,True,True,True,True,2,11.350000,22,2,600,600,600,120,600,600,1.000000,1.0,1.0,0.2,1.0
6,T01d,redacted,1800,89.95,20.0,False,False,False,False,2,18.247222,41,1,1800,1800,1800,360,1800,1800,1.000000,1.0,1.0,0.2,1.0
7,T02d,redacted,1800,89.95,20.0,False,False,False,False,3,15.281111,25,2,1800,1800,1800,360,1800,1800,1.000000,1.0,1.0,0.2,1.0
8,T03d,redacted,1800,89.95,20.0,False,False,False,False,3,36.731111,56,1,1800,1800,1800,360,1800,1800,1.000000,1.0,1.0,0.2,1.0
9,T04d,redacted,1800,89.95,20.0,False,False,False,False,3,2.827778,11,1,1800,1800,1800,360,1800,1800,1.000000,1.0,1.0,0.2,1.0


In [4]:
inventory.groupby("dataset_name")[
    [
        "frame_count",
        "duration_sec",
        "event_log_count",
        "left_images",
        "right_images",
        "driver_images",
        "depth_files",
        "label_files",
    ]
].agg(["count", "sum", "mean"])

frame_count                duration_sec               event_log_count               left_images                right_images                 \
                   count    sum    mean        count    sum   mean           count sum      mean       count    sum    mean        count    sum    mean   
dataset_name                                                                                                                                              
practice               6   3600   600.0            6  179.7  29.95               6   8  1.333333           6   3600   600.0            6   3600   600.0   
redacted              10  18000  1800.0           10  899.5  89.95              10  28  2.800000          10  17999  1799.9           10  18000  1800.0   

             driver_images                depth_files              label_files                 
                     count    sum    mean       count   sum   mean       count    sum    mean  
dataset_name                                                                                   
practice                 6   3600   600.0           6   720  120.0           6   3600   600.0  
redacted                10  18000  1800.0          10  3600  360.0          10  18000  1800.0

In [ ]:
anomalies = inventory.assign(
    missing_driver_labels=lambda df: ~df["driver_labels_available"],
    missing_risk_gt=lambda df: ~df["risk_ground_truth_available"],
    partial_road_assets=lambda df: (df["left_image_coverage"] < 1.0) | (df["right_image_coverage"] < 1.0),
    sparse_depth=lambda df: df["depth_coverage"] < 1.0,
    sparse_labels=lambda df: df["label_coverage"] < 1.0,
)
anomalies[
    [
        "trip_id",
        "dataset_name",
        "frame_count",
        "driver_labels_available",
        "risk_ground_truth_available",
        "left_image_coverage",
        "right_image_coverage",
        "driver_image_coverage",
        "depth_coverage",
        "label_coverage",
        "partial_road_assets",
        "sparse_depth",
        "sparse_labels",
    ]
]

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

inventory.plot.bar(x="trip_id", y="frame_count", color="#034EA2", ax=axes[0], legend=False)
axes[0].set_title("Frame count per trip")
axes[0].tick_params(axis="x", rotation=75)

coverage_cols = ["left_image_coverage", "driver_image_coverage", "depth_coverage", "label_coverage"]
inventory.set_index("trip_id")[coverage_cols].plot.bar(ax=axes[1])
axes[1].set_title("Modality coverage ratio")
axes[1].tick_params(axis="x", rotation=75)

inventory.groupby("dataset_name")[["driver_labels_available", "risk_ground_truth_available"]].mean().plot.bar(
    ax=axes[2], color=["#F37021", "#19226D"]
)
axes[2].set_title("Ground-truth availability")
axes[2].set_ylim(0, 1.05)
axes[2].legend(loc="lower right")

plt.tight_layout()

In [ ]:
trip_paths = nb.canonical_trip_paths()
trip_paths

In [ ]:
out_path = PROJECT_ROOT / "artifacts" / "dataset_inventory.csv"
out_path.parent.mkdir(parents=True, exist_ok=True)
inventory.to_csv(out_path, index=False)
out_path